# NEU steel-defect classifier

Thin driver notebook: all real logic lives in `src/` modules (`config.py`, `dataset.py`,
`models.py`, `train.py`, `evaluate.py`). This notebook just orchestrates so the expensive
step (loading data / training) can be run once, then evaluation cells re-run freely.

To try a different backbone, change `cfg.architecture` in the cell below to any key in
`src.models.ARCHITECTURES` (`resnet18`, `resnet34`, `efficientnet_b0`, `mobilenet_v3_small`)
and re-run from the training cells onward.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import matplotlib.pyplot as plt

from src.config import Config
from src.dataset import load_datasets, build_loaders
from src.models import DefectClassifier
from src.train import set_seed, fit
from src.evaluate import (
    classification_report_for,
    confusion_matrix_plot,
    top2_accuracy,
    compare_stage1_vs_stage2,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Load data

Run once. `files/data/train/images/` is used as-is; `files/data/validation/images/` is
stratified-split 50/50 into val/test (fixed seed, in-memory `Subset`, no augmentation on
either).

In [ ]:
cfg = Config()
cfg.architecture = "resnet18"  # <- swap the engine here
cfg.checkpoint_dir.mkdir(parents=True, exist_ok=True)

set_seed(cfg.seed)
ds = load_datasets(cfg)
train_loader, val_loader, test_loader = build_loaders(ds, cfg.batch_size)

print(f"classes: {ds.classes}")
print(f"train={len(ds.train)} val={len(ds.val)} test={len(ds.test)}")

## 2. Stage 1 — linear probe

Backbone frozen, only the classifier head trains.

In [ ]:
import torch.nn as nn
import torch.optim as optim

model = DefectClassifier(cfg.architecture, num_classes=len(ds.classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg.lr_stage1)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

stage1_ckpt = cfg.checkpoint_dir / f"{cfg.architecture}_stage1_best.pth"
history_stage1 = fit(
    model, train_loader, val_loader, optimizer, scheduler, criterion, device,
    cfg.epochs_stage1, stage1_ckpt, log_prefix="[stage1] ",
)

## 3. Stage 2 — fine-tune

Load the best stage-1 checkpoint, unfreeze the architecture's last block(s), and train with a
**fresh optimizer** over the now-trainable parameters at a lower LR (a stale optimizer here is
what silently dropped `layer4` updates in the original notebook this project is adapted from).

In [ ]:
model.load_state_dict(torch.load(stage1_ckpt, map_location=device))
model.unfreeze_finetune_block()

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg.lr_stage2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

stage2_ckpt = cfg.checkpoint_dir / f"{cfg.architecture}_stage2_finetuned.pth"
history_stage2 = fit(
    model, train_loader, val_loader, optimizer, scheduler, criterion, device,
    cfg.epochs_stage2, stage2_ckpt, log_prefix="[stage2] ",
)

## 4. Evaluate on the held-out test set

Everything below only reads `model`/`test_loader` already in memory — re-run these cells as
many times as you like without repeating steps 1–3.

In [ ]:
model.load_state_dict(torch.load(stage2_ckpt, map_location=device))

print(classification_report_for(model, test_loader, ds.classes, device))

In [ ]:
fig = confusion_matrix_plot(model, test_loader, ds.classes, device)
plt.show()

In [ ]:
print(f"top-2 accuracy (test): {top2_accuracy(model, test_loader, device):.3f}")

## 5. Stage 1 vs Stage 2 — computed live

Unlike the original notebook (which hardcoded numbers copied from a previous run), this loads
both checkpoints fresh and computes both accuracies from the actual models, on the same
held-out test set.

In [ ]:
stage1_model = DefectClassifier(cfg.architecture, num_classes=len(ds.classes)).to(device)
stage1_model.load_state_dict(torch.load(stage1_ckpt, map_location=device))

stage2_model = DefectClassifier(cfg.architecture, num_classes=len(ds.classes)).to(device)
stage2_model.unfreeze_finetune_block()
stage2_model.load_state_dict(torch.load(stage2_ckpt, map_location=device))

fig, acc_stage1, acc_stage2 = compare_stage1_vs_stage2(stage1_model, stage2_model, test_loader, device)
plt.show()
print(f"stage1 acc={acc_stage1:.3f}  stage2 acc={acc_stage2:.3f}")

## 6. Swap the engine

Change the architecture and re-run the whole pipeline headlessly for comparison, without
touching any other code.

In [ ]:
from src.train import run_two_stage_training

cfg_alt = Config()
cfg_alt.architecture = "efficientnet_b0"  # try: resnet34, efficientnet_b0, mobilenet_v3_small

result = run_two_stage_training(cfg_alt)

alt_model = DefectClassifier(cfg_alt.architecture, num_classes=len(result["classes"])).to(device)
alt_model.unfreeze_finetune_block()
alt_model.load_state_dict(torch.load(result["stage2_checkpoint"], map_location=device))

print(classification_report_for(alt_model, result["test_loader"], result["classes"], device))

## 7. Generate PDF report

Builds a multi-page PDF (title/summary, training curves, confusion matrix,
classification report table, stage1-vs-stage2 comparison) from the checkpoints
and history already produced above.

In [ ]:
from src.report import generate_report

report_path = cfg.checkpoint_dir.parent / "reports" / f"{cfg.architecture}_report.pdf"
generate_report(cfg, report_path)
print(f"Report written to {report_path}")